# Bài 3.9 · Attention bên trong GPT-2

**Câu hỏi của buổi này:** attention trong một model thật trông thế nào? Ta mở **GPT-2** (124 triệu tham số, 12 tầng, mỗi tầng 12 head), tự tính lại một head bằng công thức của bài, rồi xem các head học được gì và attention tốn bao nhiêu khi câu dài ra.

| Mục | Sẽ thấy |
|---|---|
| 1. Câu và token | bảng token của một câu |
| 2. Tính tay một head | Q, K, V, ma trận attention tự tính khớp với model |
| 3. Vì sao chia √d | độ nhọn của softmax khi có và không chia |
| 4. Causal mask | ma trận tam giác dưới; đổi token sau không làm đổi đầu ra trước |
| 5. Các head nhìn gì | head nhìn token trước, head nhìn token đầu, head nối "it" với danh từ |
| 6. Attention không phải lời giải thích | thay token rồi đo dự đoán đổi bao nhiêu |
| 7. Thông tin vị trí | độ giống nhau giữa các vector vị trí |
| 8. Chi phí theo độ dài | thời gian và bộ nhớ khi câu dài gấp đôi |
| 9. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 2 phút. Bài giảng: [ai-canuc80k · bài 3.9](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/3.9).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style
    if fmt:
        st = st.format(fmt)
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import time, math
import torch, transformers
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
transformers.logging.set_verbosity_error()

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
with quiet():
    tok = GPT2TokenizerFast.from_pretrained('gpt2')
    lm = GPT2LMHeadModel.from_pretrained('gpt2', attn_implementation='eager').to(device).eval()
gpt = lm.transformer
cfg = lm.config
table(pd.DataFrame([('torch', torch.__version__), ('transformers', transformers.__version__),
                    ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'),
                    ('GPT-2', f'{sum(p.numel() for p in lm.parameters()) / 1e6:.0f} triệu tham số, {cfg.n_layer} tầng, {cfg.n_head} head, d = {cfg.n_embd}')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Câu và token

GPT-2 cắt câu thành **token** (từ hoặc mảnh từ; dấu cách dính vào đầu token, in ra là `Ġ`). Mỗi token thành một vector 768 số: vector của token cộng vector của vị trí.

In [ ]:
text = 'The cat sat on the mat because it was tired.'
ids = tok(text, return_tensors='pt').input_ids.to(device)
toks = [t.replace('Ġ', '·') for t in tok.convert_ids_to_tokens(ids[0])]
L = len(toks)
table(pd.DataFrame([toks], columns=[f'{i}' for i in range(L)], index=['token']), f'Câu có {L} token (· là dấu cách)', index=True)
with torch.no_grad():
    out = gpt(ids, output_attentions=True, output_hidden_states=True)
att = torch.stack(out.attentions)[:, 0].cpu()          # (tầng, head, L, L)
check(f'attention có shape {tuple(att.shape)} = (tầng, head, token hỏi, token được nhìn)', tuple(att.shape) == (cfg.n_layer, cfg.n_head, L, L))
check('mỗi hàng attention cộng lại bằng 1', torch.allclose(att.sum(-1), torch.ones(1), atol=1e-4))

## 2. Tính tay một head

Ở tầng 0, đầu vào $X$ (sau LayerNorm) được chiếu thành $Q = XW_Q$, $K = XW_K$, $V = XW_V$; mỗi head dùng 64 trong 768 cột. Rồi

$$A = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right), \qquad \text{đầu ra} = AV,$$

với $M$ là mask: $-\infty$ ở các ô token nhìn về tương lai. Tự tính $A$ cho head 0 rồi so với ma trận model trả về.

In [ ]:
blk = gpt.h[0]
x = blk.ln_1(out.hidden_states[0])
q, k, v = blk.attn.c_attn(x).split(cfg.n_embd, dim=-1)
dk = cfg.n_embd // cfg.n_head
H = 0
qh, kh, vh = [t[0, :, H * dk:(H + 1) * dk] for t in (q, k, v)]
scores = qh @ kh.T / math.sqrt(dk)
mask = torch.triu(torch.ones(L, L, dtype=torch.bool, device=device), 1)
A = scores.masked_fill(mask, float('-inf')).softmax(-1).detach().cpu()

table(pd.DataFrame([('X', tuple(x.shape[1:])), ('Q, K, V của một head', tuple(qh.shape)), ('QKᵀ / √d', tuple(scores.shape)), ('A', tuple(A.shape))],
                   columns=['đại lượng', 'shape']), f'Head {H}, tầng 0 (d_k = {dk})')
diff = (A - att[0, H]).abs().max().item()
check(f'A tự tính khớp attention của model (chênh lớn nhất {diff:.1e})', diff < 1e-4)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for ax, M, t in [(axes[0], scores.detach().cpu().masked_fill(mask.cpu(), float('nan')), 'điểm QKᵀ/√d (ô trắng: bị che)'), (axes[1], A, 'A = softmax theo từng hàng')]:
    im = ax.imshow(M, cmap='Blues'); ax.grid(False); plt.colorbar(im, ax=ax, fraction=.046)
    ax.set_xticks(range(L), toks, rotation=60, ha='right'); ax.set_yticks(range(L), toks)
    ax.set_xlabel('token được nhìn (key)'); ax.set_ylabel('token đang hỏi (query)'); ax.set_title(t)
plt.tight_layout(); plt.show()

## 3. Vì sao chia √d

Tích vô hướng của hai vector 64 chiều có độ lớn tăng theo $\sqrt{d}$. Không chia thì điểm lớn, softmax dồn gần hết trọng số vào một ô (bão hòa), gradient qua các ô còn lại gần 0. Đo trên **mọi head của tầng 0**: trọng số lớn nhất của mỗi hàng.

In [ ]:
qa, ka = [t[0].view(L, cfg.n_head, dk).transpose(0, 1) for t in (q, k)]
raw = (qa @ ka.transpose(1, 2)).detach()
rows = {}
for name, s in [('chia √d (đúng)', raw / math.sqrt(dk)), ('không chia', raw)]:
    a = s.masked_fill(mask, float('-inf')).softmax(-1)[:, 3:]      # bỏ 3 hàng đầu vì ít ô để chọn
    rows[name] = a.max(-1).values.flatten().cpu().numpy()

fig, ax = plt.subplots(figsize=(9, 3.6))
bins = np.linspace(0, 1, 26)
for (name, v), c in zip(rows.items(), [C['blue'], C['clay']]):
    ax.hist(v, bins=bins, alpha=.7, color=c, label=f'{name}: trung bình {v.mean():.2f}')
ax.set(xlabel='trọng số lớn nhất trong một hàng attention', ylabel='số hàng', title=f'Tầng 0, {cfg.n_head} head: không chia √{dk} thì softmax dồn về một ô'); ax.legend()
plt.tight_layout(); plt.show()
sat = {n: (v > 0.95).mean() for n, v in rows.items()}
note(f"Hàng có một ô chiếm trên 95% trọng số: <b>{sat['chia √d (đúng)']:.0%}</b> khi chia √d, <b>{sat['không chia']:.0%}</b> khi không chia.")

## 4. Causal mask

GPT-2 đoán token tiếp theo, nên token ở vị trí $i$ chỉ được nhìn các vị trí $\le i$. Hai cách kiểm: mọi ô phía trên đường chéo bằng 0; và đổi một token ở cuối câu thì đầu ra của các token phía trước không đổi.

In [ ]:
upper = att.masked_select(torch.triu(torch.ones(L, L, dtype=torch.bool), 1)).abs().max().item()
check(f'mọi ô trên đường chéo, mọi tầng, mọi head đều bằng 0 (lớn nhất {upper:.1e})', upper == 0)

ids2 = ids.clone(); j = L - 3
ids2[0, j] = tok(' happy').input_ids[0]
with torch.no_grad():
    h1, h2 = gpt(ids).last_hidden_state[0], gpt(ids2).last_hidden_state[0]
change = (h1 - h2).norm(dim=-1).cpu().numpy()
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.bar(range(L), change, color=[C['gray'] if i < j else C['clay'] for i in range(L)])
ax.set_xticks(range(L), toks, rotation=45, ha='right'); ax.set_ylabel('độ lệch đầu ra tầng cuối')
ax.set_title(f'Đổi token {j} ("{toks[j]}" → "·happy"): chỉ token từ vị trí {j} trở đi bị ảnh hưởng')
plt.tight_layout(); plt.show()
check(f'các token trước vị trí {j} không đổi (lệch lớn nhất {change[:j].max():.1e})', change[:j].max() < 1e-3)

## 5. Các head nhìn gì

144 head không làm cùng một việc. Tìm tự động ba kiểu dễ thấy:

- head **nhìn token ngay trước**: trọng số trung bình đặt vào ô $(i, i-1)$ cao nhất;
- head **dồn về token đầu**: trọng số đặt vào cột 0 cao nhất (nhiều model dùng token đầu như chỗ "đỗ" khi không cần nhìn đâu);
- head mà token **"it"** nhìn vào **"cat"** nhiều nhất.

In [ ]:
idx = torch.arange(1, L)
prev = att[:, :, idx, idx - 1].mean(-1)
first = att[:, :, 2:, 0].mean(-1)
i_it, i_cat = toks.index('·it'), toks.index('·cat')
coref = att[:, :, i_it, i_cat]
picks = [('nhìn token ngay trước', prev), ('dồn về token đầu', first), ('"it" nhìn "cat"', coref)]

fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))
found = []
for ax, (name, score) in zip(axes, picks):
    l, h = divmod(int(score.argmax()), cfg.n_head)
    found.append((name, f'tầng {l}, head {h}', float(score[l, h])))
    ax.imshow(att[l, h], cmap='Blues', vmin=0, vmax=1); ax.grid(False)
    ax.set_xticks(range(L), toks, rotation=60, ha='right', fontsize=8); ax.set_yticks(range(L), toks, fontsize=8)
    ax.set_title(f'{name}\n(tầng {l}, head {h})', fontsize=11)
plt.tight_layout(); plt.show()
table(pd.DataFrame(found, columns=['kiểu head', 'vị trí', 'điểm']), 'Ba head tìm được trong 144 head', fmt={'điểm': '{:.2f}'})

fig, ax = plt.subplots(figsize=(10, 3.4))
im = ax.imshow(coref.T, cmap='Blues', aspect='auto'); plt.colorbar(im, ax=ax, fraction=.03)
ax.set(xlabel='tầng', ylabel='head', title='Trọng số "it" → "cat" ở từng tầng, từng head'); ax.grid(False)
ax.set_xticks(range(cfg.n_layer)); ax.set_yticks(range(cfg.n_head))
plt.tight_layout(); plt.show()
note(f"\"it\" nhìn \"cat\" mạnh nhất ở {found[2][1]} ({found[2][2]:.2f}); trung bình trên cả 144 head chỉ {coref.mean():.2f}. Mỗi head chuyên một việc, phần lớn head không làm việc này.")

## 6. Attention không phải lời giải thích

Trọng số attention chỉ cho biết một head **trộn** những vector nào. Muốn biết token nào thật sự ảnh hưởng tới dự đoán, phải **thay token** rồi đo dự đoán đổi bao nhiêu. Ở đây đo xác suất token tiếp theo sau "…because it was" khi thay từng token phía trước bằng `·the` (ô vốn đã là `·the` thì không đổi gì).

In [ ]:
base = tok('The cat sat on the mat because it was', return_tensors='pt').input_ids.to(device)
bt = [t.replace('Ġ', '·') for t in tok.convert_ids_to_tokens(base[0])]
with torch.no_grad():
    p0 = lm(base).logits[0, -1].softmax(-1)
top_ids = p0.topk(5).indices
table(pd.DataFrame({'token tiếp theo': [tok.decode(int(t)) for t in top_ids], 'xác suất': p0[top_ids].cpu().numpy()}), f'GPT-2 đoán gì sau "{tok.decode(base[0])}"', fmt={'xác suất': '{:.3f}'})

the = tok(' the').input_ids[0]; shift = []
for pos in range(len(bt) - 1):
    b2 = base.clone(); b2[0, pos] = the
    with torch.no_grad():
        p = lm(b2).logits[0, -1].softmax(-1)
    shift.append(0.5 * (p - p0).abs().sum().item())
with torch.no_grad():
    a_last = torch.stack(lm(base, output_attentions=True).attentions)[:, 0, :, -1, :-1].mean((0, 1)).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
axes[0].bar(bt[:-1], a_last, color=C['blue']); axes[0].set_title('attention trung bình từ "·was" (mọi tầng, mọi head)')
axes[1].bar(bt[:-1], shift, color=C['clay']); axes[1].set_title('dự đoán đổi bao nhiêu khi thay token đó bằng "·the"')
for ax in axes: ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()
corr = np.corrcoef(a_last, shift)[0, 1]
note(f"Token được nhìn nhiều nhất: <b>{bt[int(np.argmax(a_last))]}</b>; token thay vào làm dự đoán đổi nhiều nhất: <b>{bt[int(np.argmax(shift))]}</b>. "
     f"Tương quan giữa hai cột chỉ {corr:.2f}: không đọc attention như lời giải thích.")

## 7. Thông tin vị trí

Attention tự nó không biết thứ tự: đảo các token thì các cặp điểm $q_i^\top k_j$ chỉ đổi chỗ. GPT-2 cộng vào mỗi token một **vector vị trí** đã học (1024 vị trí). Độ giống cosine giữa các vector vị trí cho thấy model đã học được "gần nhau".

In [ ]:
P = gpt.wpe.weight[:256].detach().float().cpu()
Pn = P / P.norm(dim=1, keepdim=True)
sim = (Pn @ Pn.T).numpy()
fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1, 1.2]})
im = axes[0].imshow(sim, cmap='RdBu_r', vmin=-1, vmax=1); axes[0].grid(False); plt.colorbar(im, ax=axes[0], fraction=.046)
axes[0].set(xlabel='vị trí', ylabel='vị trí', title='Cosine giữa 256 vector vị trí đầu')
for p0_, c in zip([20, 100, 200], [C['blue'], C['sand'], C['clay']]):
    axes[1].plot(sim[p0_], color=c, label=f'so với vị trí {p0_}'); axes[1].axvline(p0_, color=c, ls=':', lw=1)
axes[1].set(xlabel='vị trí', ylabel='cosine', title='Vị trí gần nhau thì vector giống nhau'); axes[1].legend()
plt.tight_layout(); plt.show()
near, far = np.mean([sim[i, i + 1] for i in range(1, 255)]), np.mean([sim[i, i + 100] for i in range(1, 155)])
check('hai vị trí kề nhau giống nhau hơn hai vị trí cách 100', near > far)
note(f'Cosine trung bình của hai vị trí kề nhau: <b>{near:.3f}</b>; cách nhau 100 vị trí: {far:.2f}.')

## 8. Chi phí theo độ dài

Ma trận attention có $L \times L$ ô, nên câu dài gấp đôi thì số ô gấp bốn. Đo trên một tầng attention 12 head, batch 1, với hai cách: tính đủ ma trận (`softmax(QKᵀ)V`) và `scaled_dot_product_attention` của PyTorch (không lưu đủ ma trận khi có kernel tối ưu).

In [ ]:
def bench(Lx, fused, reps=5):
    qx = torch.randn(1, cfg.n_head, Lx, dk, device=device, dtype=torch.float16 if device == 'cuda' else torch.float32)
    kx, vx = torch.randn_like(qx), torch.randn_like(qx)
    def run():
        if fused: return F.scaled_dot_product_attention(qx, kx, vx, is_causal=True)
        s = (qx @ kx.transpose(-1, -2)) / math.sqrt(dk)
        s = s.masked_fill(torch.triu(torch.ones(Lx, Lx, dtype=torch.bool, device=device), 1), float('-inf'))
        return s.softmax(-1) @ vx
    if device == 'cuda': torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    base_mem = torch.cuda.memory_allocated() if device == 'cuda' else 0
    run()
    if device == 'cuda': torch.cuda.synchronize()
    t = time.time()
    for _ in range(reps): run()
    if device == 'cuda': torch.cuda.synchronize()
    mem = (torch.cuda.max_memory_allocated() - base_mem) / 2**20 if device == 'cuda' else np.nan
    return (time.time() - t) / reps * 1000, mem

Ls = [512, 1024, 2048, 4096, 8192] if device == 'cuda' else [128, 256, 512, 1024]
rows = [(Lx, name, *bench(Lx, fused)) for Lx in Ls for name, fused in [('tính đủ ma trận', False), ('scaled_dot_product_attention', True)]]
cost = pd.DataFrame(rows, columns=['L', 'cách', 'ms', 'MB thêm'])
piv = cost.pivot(index='L', columns='cách', values='ms').reset_index()
table(piv, 'Thời gian một lần attention (ms)', fmt={c: '{:.2f}' for c in piv.columns if c != 'L'})

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for (name, g), c in zip(cost.groupby('cách', sort=False), [C['clay'], C['blue']]):
    axes[0].plot(g['L'], g['ms'], marker='o', color=c, label=name)
    axes[1].plot(g['L'], g['MB thêm'], marker='o', color=c, label=name)
for ax, t in zip(axes, ['thời gian (ms)', 'bộ nhớ thêm (MB)']):
    if ax is axes[1] and cost['MB thêm'].isna().all():
        ax.axis('off'); continue                      # CPU: không đo bộ nhớ
    ax.set_xscale('log', base=2); ax.set_yscale('log'); ax.set(xlabel='độ dài L', title=t); ax.legend()
plt.tight_layout(); plt.show()
full = cost[cost['cách'] == 'tính đủ ma trận'].set_index('L')
r_t = full['ms'].iloc[-1] / full['ms'].iloc[-2]
note(f"Tính đủ ma trận: L từ {Ls[-2]} lên {Ls[-1]} (gấp đôi) thì thời gian gấp <b>{r_t:.1f}</b> lần"
     + (f", bộ nhớ gấp {full['MB thêm'].iloc[-1] / full['MB thêm'].iloc[-2]:.1f} lần." if device == 'cuda' else '.'))

## 9. Tóm tắt

In [ ]:
summary = [('A tự tính khớp model (chênh lớn nhất)', f'{diff:.1e}'),
           ('hàng bão hòa > 0.95: chia √d / không chia', f"{sat['chia √d (đúng)']:.0%} / {sat['không chia']:.0%}"),
           ('ô trên đường chéo lớn nhất (causal mask)', f'{upper:.1e}'),
           ('head "it" → "cat" mạnh nhất', f'{found[2][1]}: {found[2][2]:.2f}'),
           ('tương quan attention và mức ảnh hưởng', f'{corr:.2f}'),
           (f'thời gian khi L gấp đôi ({Ls[-2]} → {Ls[-1]})', f'×{r_t:.1f}')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** đổi câu ở mục 1 thành "The trophy did not fit in the suitcase because it was too big." rồi tìm head nối "it" với "trophy"; đổi `H = 0` ở mục 2 sang head khác; ở mục 6 thay bằng token khác `·the`.